# Proyecto Transformer — Traveler Insights UPTC 2026

**Metodología:** CRISP-ML(Q)  
**Integrantes:** _Completar_  
**Roles iniciales:** _Completar y rotar: datos, modelado, evaluación, documentación_  
**Repositorio:** _Completar_  
**Equipo/perfil de Kaggle:** _Completar_

> Objetivo del cuaderno: ejecutar un flujo reproducible desde la carga de datos hasta la generación del archivo de envío, dejando evidencia de riesgos, QA y decisiones.

## Cómo usar este cuaderno

1. Ejecutar las celdas en orden.
2. Completar todos los campos marcados como **PENDIENTE**.
3. No usar el conjunto etiquetado de prueba para seleccionar hiperparámetros.
4. Registrar cada corrida; la rúbrica exige al menos cinco.
5. Conservar las salidas, tablas y figuras que sirvan como evidencia para el informe de 10–15 páginas.

**Métrica de trabajo provisional:** F1 macro. Debe reemplazarse si la competencia define otra métrica oficial.

## Fase 0 — Portada y resumen

### Resumen (máximo 200 palabras)

**PENDIENTE:** redactar al finalizar. Debe incluir problema, modelo, métrica oficial en validación local y leaderboard público, y una limitación.

### Control de entregables

- [ ] Notebook reproducible de principio a fin
- [ ] Documento técnico CRISP-ML(Q), 10–15 páginas
- [ ] Registro de al menos 5 corridas
- [ ] Bitácora de IA generativa
- [ ] Envío final de Kaggle
- [ ] Demo de inferencia
- [ ] Evidencia de análisis manual de al menos 20 errores

## Configuración reproducible

In [ ]:
# En Kaggle/Colab, descomentar si hace falta:
# %pip install -q transformers datasets accelerate evaluate sentencepiece captum

from pathlib import Path
import hashlib
import json
import os
import platform
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path.cwd()
DATA_DIR = ROOT / "Data"
ARTIFACTS_DIR = ROOT / "artifacts"
MODELS_DIR = ROOT / "models"
ARTIFACTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "submission.csv"

ID_COL = "ID"
TEXT_COL = "Comentario"
TARGET_COL = "Sentimiento"
RATING_COL = "Valoración_num"
OFFICIAL_METRIC = "f1_macro"  # PENDIENTE: confirmar en las reglas de Kaggle

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Proyecto:", ROOT.resolve())
print("Semilla:", SEED)

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
submission_df = pd.read_csv(SUBMISSION_PATH)

print("train:", train_df.shape)
print("test etiquetado:", test_df.shape)
print("submission sin etiqueta:", submission_df.shape)
display(train_df.head(3))

required = {ID_COL, TEXT_COL, TARGET_COL}
assert required.issubset(train_df.columns)
assert required.issubset(test_df.columns)
assert required.issubset(submission_df.columns)
assert train_df[ID_COL].is_unique
assert test_df[ID_COL].is_unique
assert submission_df[ID_COL].is_unique
assert train_df[TARGET_COL].notna().all()
assert test_df[TARGET_COL].notna().all()
assert submission_df[TARGET_COL].isna().all()

LABELS = sorted(train_df[TARGET_COL].unique())
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}
print("Etiquetas:", LABELS)

## Fase 1 — Comprensión del negocio y de los datos

### Objetivos

- **Negocio — PENDIENTE:** identificar quién usará el clasificador y qué decisión apoyará.
- **ML:** clasificar una reseña turística escrita en español en negativo, neutral o positivo.
- **Entrada principal:** texto de la columna Comentario.
- **Salida:** etiqueta Sentimiento.
- **Éxito ML — PENDIENTE:** F1 macro del Transformer ≥ F1 macro del baseline + margen definido por el grupo.
- **Éxito económico — PENDIENTE:** límite de latencia, memoria y tamaño del modelo.
- **Restricciones — PENDIENTE:** GPU, tiempo, licencia y límites de Kaggle/Colab.

### Riesgo crítico identificado

La columna Valoración_num está prácticamente asociada de forma determinista con Sentimiento. Incluirla puede producir fuga conceptual o un atajo que no evalúa comprensión del texto. El modelo principal y el baseline usarán únicamente Comentario. La relación se documenta como hallazgo de calidad; no se explota para inflar la métrica.

El archivo test.csv tiene etiquetas y se reserva como evaluación final. El archivo submission.csv tiene la etiqueta vacía y es el conjunto para generar el envío.

In [ ]:
def resumen_calidad(df, nombre):
    return pd.Series({
        "conjunto": nombre,
        "filas": len(df),
        "textos_vacíos": df[TEXT_COL].isna().sum() + df[TEXT_COL].fillna("").str.strip().eq("").sum(),
        "comentarios_duplicados": df[TEXT_COL].duplicated().sum(),
        "ids_duplicados": df[ID_COL].duplicated().sum(),
        "etiquetas_vacías": df[TARGET_COL].isna().sum(),
    })

display(pd.DataFrame([
    resumen_calidad(train_df, "train"),
    resumen_calidad(test_df, "test"),
    resumen_calidad(submission_df, "submission"),
]))

display(train_df[TARGET_COL].value_counts().rename("conteo").to_frame())
display(pd.crosstab(train_df[RATING_COL], train_df[TARGET_COL]))

normalized_train = train_df[TEXT_COL].fillna("").str.strip().str.lower()
normalized_test = test_df[TEXT_COL].fillna("").str.strip().str.lower()
normalized_sub = submission_df[TEXT_COL].fillna("").str.strip().str.lower()

print("Solapamientos exactos train–test:", normalized_train.isin(set(normalized_test)).sum())
print("Solapamientos exactos train–submission:", normalized_train.isin(set(normalized_sub)).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = train_df[TARGET_COL].value_counts().index
sns.countplot(data=train_df, x=TARGET_COL, order=order, ax=axes[0])
axes[0].set_title("Distribución de clases — train")
axes[0].set_xlabel("Sentimiento")
axes[0].set_ylabel("Cantidad")

char_lengths = train_df[TEXT_COL].fillna("").str.len()
sns.histplot(char_lengths, bins=35, ax=axes[1])
axes[1].set_title("Longitud de comentarios en caracteres")
axes[1].set_xlabel("Caracteres")
plt.tight_layout()
plt.show()

display(train_df.groupby(TARGET_COL)[TEXT_COL].agg(
    cantidad="size",
    caracteres_mediana=lambda s: s.str.len().median(),
    palabras_mediana=lambda s: s.str.split().str.len().median(),
))

### QA de la fase 1

| Riesgo | Método QA | Evidencia esperada |
|---|---|---|
| Desbalance fuerte | Conteos y proporciones por clase; F1 macro | Tabla/gráfico de clases |
| Fuga por metadatos | Entrenar solo con Comentario; auditar relación valoración–etiqueta | Tabla cruzada |
| Duplicados entre particiones | Agrupar por hash del texto antes de dividir | Aserción sin solapamiento |
| Ruido o etiquetas dudosas | Revisión estratificada de ejemplos | Tabla de casos |
| Datos personales | Búsqueda de correos, teléfonos y nombres; anonimización si corresponde | Conteos y ejemplos anonimizados |

## Fase 2 — Ingeniería de datos

Se conserva el texto casi crudo. No se eliminan tildes, signos, mayúsculas ni palabras vacías porque pueden aportar información al Transformer. Para el baseline TF‑IDF solo se aplica la normalización interna indicada en su vectorizador.

In [ ]:
def normalize_for_grouping(text):
    return " ".join(str(text).lower().strip().split())

train_df = train_df.copy()
train_df["text_group"] = train_df[TEXT_COL].map(
    lambda x: hashlib.sha256(normalize_for_grouping(x).encode("utf-8")).hexdigest()
)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, valid_idx = next(splitter.split(
    train_df,
    y=train_df[TARGET_COL],
    groups=train_df["text_group"]
))

dev_df = train_df.iloc[train_idx].reset_index(drop=True)
valid_df = train_df.iloc[valid_idx].reset_index(drop=True)

assert set(dev_df["text_group"]).isdisjoint(set(valid_df["text_group"]))
assert set(dev_df[TARGET_COL]) == set(valid_df[TARGET_COL]) == set(LABELS)
assert dev_df[TEXT_COL].notna().all() and valid_df[TEXT_COL].notna().all()

print("Desarrollo:", dev_df.shape)
print("Validación:", valid_df.shape)
display(pd.DataFrame({
    "desarrollo": dev_df[TARGET_COL].value_counts(normalize=True),
    "validación": valid_df[TARGET_COL].value_counts(normalize=True),
}).fillna(0).round(4))

### Tokenización y elección de max_length

Ejecutar la siguiente celda para cada checkpoint candidato. Elegir max_length con evidencia: percentil de longitudes y porcentaje de textos truncados, equilibrando cobertura, memoria y latencia.

In [ ]:
CHECKPOINTS = [
    "dccuchile/bert-base-spanish-wwm-cased",
    "xlm-roberta-base",
]
MODEL_NAME = CHECKPOINTS[0]
MAX_LENGTH = 256  # PENDIENTE: reemplazar después del análisis
print("Candidatos:", CHECKPOINTS)
print("Checkpoint activo:", MODEL_NAME)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
token_lengths = train_df[TEXT_COL].map(
    lambda text: len(tokenizer(str(text), add_special_tokens=True, truncation=False)["input_ids"])
)

display(token_lengths.describe(percentiles=[.5, .75, .9, .95, .99]).to_frame("tokens"))
for candidate in [64, 128, 192, 256, 384, 512]:
    print(f"max_length={candidate:>3}: {(token_lengths > candidate).mean():.2%} truncado")

sns.histplot(token_lengths, bins=40)
plt.axvline(MAX_LENGTH, color="red", linestyle="--", label=f"max_length={MAX_LENGTH}")
plt.legend()
plt.title("Longitud con el tokenizer del checkpoint activo")
plt.show()

### QA de la fase 2

- La partición usa grupos por texto normalizado para evitar duplicados entre desarrollo y validación.
- Las etiquetas están dentro del conjunto esperado.
- La semilla es fija.
- **PENDIENTE:** justificar el tratamiento del desbalance.
- **PENDIENTE:** registrar el porcentaje truncado y la decisión final de max_length.
- Si se aumenta información, documentar método, proporción y revisión manual de calidad.

## Fase 3 — Ingeniería del modelo y diseño experimental

### 3.1 Baseline clásico obligatorio

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents=None,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=40_000,
    )),
    ("clf", LogisticRegression(
        max_iter=2_000,
        class_weight="balanced",
        random_state=SEED,
    )),
])

start = time.perf_counter()
baseline.fit(dev_df[TEXT_COL], dev_df[TARGET_COL])
baseline_seconds = time.perf_counter() - start
valid_pred_baseline = baseline.predict(valid_df[TEXT_COL])
baseline_f1 = f1_score(valid_df[TARGET_COL], valid_pred_baseline, average="macro")

print(f"F1 macro baseline: {baseline_f1:.4f}")
print(f"Tiempo de ajuste: {baseline_seconds:.2f} s")
print(classification_report(valid_df[TARGET_COL], valid_pred_baseline, digits=4))

### 3.2 Comparación de checkpoints

| Checkpoint | Idioma/preentrenamiento | Tamaño | Licencia verificada | Motivo | Resultado |
|---|---|---:|---|---|---|
| dccuchile/bert-base-spanish-wwm-cased | Español | PENDIENTE | PENDIENTE | Modelo monolingüe | PENDIENTE |
| xlm-roberta-base | Multilingüe | PENDIENTE | PENDIENTE | Comparador multilingüe | PENDIENTE |

La licencia y los detalles deben verificarse en la tarjeta oficial de cada modelo antes del entrenamiento.

### 3.3 Fine-tuning del Transformer

La siguiente plantilla deja explícitos encoder, cabeza de clasificación, hiperparámetros, early stopping y métricas. Ejecutarla una vez completada la selección de max_length.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

def to_hf_dataset(df):
    tmp = df[[TEXT_COL, TARGET_COL]].copy()
    tmp["label"] = tmp[TARGET_COL].map(label2id)
    return Dataset.from_pandas(tmp[[TEXT_COL, "label"]], preserve_index=False)

hf_dev = to_hf_dataset(dev_df)
hf_valid = to_hf_dataset(valid_df)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )

hf_dev = hf_dev.map(tokenize_batch, batched=True)
hf_valid = hf_valid.map(tokenize_batch, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
)

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "run_001"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_dev,
    eval_dataset=hf_valid,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Entrenamiento costoso: ejecutar con GPU después de revisar la configuración.
# trainer.train()
# trainer.save_model(MODELS_DIR / "best_transformer")
# tokenizer.save_pretrained(MODELS_DIR / "best_transformer")

### 3.4 Registro obligatorio de experimentos

Cambiar una decisión principal por corrida y registrar al menos cinco: baseline, checkpoint A, checkpoint B y dos mejoras justificadas.

In [ ]:
EXPERIMENT_LOG = ARTIFACTS_DIR / "experimentos.csv"
EXPERIMENT_COLUMNS = [
    "run_id", "fecha", "modelo", "cambio", "seed", "max_length",
    "learning_rate", "batch_size", "epocas", "metrica", "valor",
    "tiempo_s", "hardware", "notas"
]

def registrar_experimento(**row):
    if EXPERIMENT_LOG.exists():
        log = pd.read_csv(EXPERIMENT_LOG)
    else:
        log = pd.DataFrame(columns=EXPERIMENT_COLUMNS)
    record = {column: row.get(column, None) for column in EXPERIMENT_COLUMNS}
    log = pd.concat([log, pd.DataFrame([record])], ignore_index=True)
    log.to_csv(EXPERIMENT_LOG, index=False)
    return log

# Ejemplo: descomentar una sola vez para registrar el baseline.
# registrar_experimento(
#     run_id="baseline_001",
#     fecha=pd.Timestamp.now().isoformat(timespec="seconds"),
#     modelo="TF-IDF + LogisticRegression",
#     cambio="Baseline obligatorio",
#     seed=SEED,
#     metrica=OFFICIAL_METRIC,
#     valor=baseline_f1,
#     tiempo_s=baseline_seconds,
#     hardware=platform.platform(),
#     notas="Solo texto; class_weight=balanced",
# )

### QA de la fase 3

| Riesgo | Control |
|---|---|
| Sobreajuste a validación o leaderboard | Split fijo, test reservado y límite de envíos usados para decidir |
| Corridas no comparables | Mismo split, misma métrica y cambio principal documentado |
| Resultados no trazables | CSV de experimentos, semillas, versiones y hardware |
| Elección oportunista | Criterios definidos antes de consultar test/leaderboard |
| Sobreajuste durante fine-tuning | Curvas train/validación y early stopping |

## Fase 4 — Evaluación del modelo

La evaluación final debe incluir métrica oficial, F1 por clase y macro, matriz de confusión, incertidumbre, al menos 20 errores revisados, robustez, responsabilidad y una decisión explícita.

In [ ]:
def evaluar(y_true, y_pred, labels=LABELS, titulo="Modelo"):
    macro = f1_score(y_true, y_pred, average="macro")
    report = pd.DataFrame(
        classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    ).T
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"{titulo} — F1 macro: {macro:.4f}")
    display(report)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title(f"Matriz de confusión — {titulo}")
    plt.show()
    return macro, report, cm

_ = evaluar(valid_df[TARGET_COL], valid_pred_baseline, titulo="Baseline")

In [ ]:
def bootstrap_f1_ci(y_true, y_pred, n_boot=2000, confidence=0.95, seed=SEED):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < len(np.unique(y_true)):
            continue
        scores.append(f1_score(y_true[idx], y_pred[idx], average="macro"))
    alpha = 1 - confidence
    return np.quantile(scores, [alpha / 2, 1 - alpha / 2])

ci_low, ci_high = bootstrap_f1_ci(valid_df[TARGET_COL], valid_pred_baseline)
print(f"IC bootstrap 95% del F1 macro: [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
error_analysis = valid_df[[ID_COL, TEXT_COL, TARGET_COL, "Sitio", "Municipio"]].copy()
error_analysis["prediccion"] = valid_pred_baseline
error_analysis = error_analysis[error_analysis[TARGET_COL] != error_analysis["prediccion"]].copy()
error_analysis["categoria_error"] = ""
error_analysis["hipotesis_causa"] = ""
error_analysis["accion_o_mitigacion"] = ""

manual_errors = error_analysis.groupby(TARGET_COL, group_keys=False).head(10).head(20)
manual_errors.to_csv(ARTIFACTS_DIR / "analisis_20_errores.csv", index=False)
display(manual_errors)

print("Completar manualmente al menos 20 filas con categorías como:")
print("ironía, negación, reseña mixta, regionalismo, error ortográfico o posible error de etiqueta")

### Robustez, explicabilidad y responsabilidad

- **Robustez:** crear subconjuntos de textos largos, textos con ruido ortográfico y, si es posible, variantes regionales; reportar degradación frente al conjunto general.
- **Explicabilidad:** aplicar SHAP o Integrated Gradients a ejemplos representativos. Aclarar que atención no equivale a explicación.
- **Sesgos:** comparar resultados por Sitio, Municipio, longitud y registro lingüístico, cuidando tamaños muestrales.
- **Privacidad:** revisar posibles datos personales antes de publicar ejemplos.
- **Decisión — PENDIENTE:** indicar Sí/No frente a cada criterio de éxito definido en la fase 1.

## Fase 5 — Despliegue adaptado

Después de guardar el mejor modelo, el pipeline debe cargarlo desde disco, medir costo y generar el envío sin reentrenar.

In [ ]:
def medir_inferencia(predict_fn, textos):
    start = time.perf_counter()
    predicciones = predict_fn(list(textos))
    elapsed = time.perf_counter() - start
    return predicciones, {
        "ejemplos": len(textos),
        "tiempo_total_s": elapsed,
        "latencia_media_ms": 1000 * elapsed / max(len(textos), 1),
    }

# Para el baseline; sustituir por el Transformer guardado en el envío oficial.
submission_pred, latency = medir_inferencia(
    baseline.predict,
    submission_df[TEXT_COL].fillna("")
)
print(latency)

kaggle_submission = submission_df.copy()
kaggle_submission[TARGET_COL] = submission_pred
assert kaggle_submission[TARGET_COL].notna().all()
assert len(kaggle_submission) == len(submission_df)
assert set(kaggle_submission[TARGET_COL]).issubset(set(LABELS))

OUTPUT_PATH = ARTIFACTS_DIR / "submission_predictions.csv"
kaggle_submission.to_csv(OUTPUT_PATH, index=False)
print("Archivo generado:", OUTPUT_PATH.resolve())
display(kaggle_submission[[ID_COL, TARGET_COL]].head())

In [ ]:
def clasificar_resena(texto, predict_fn=baseline.predict):
    texto = str(texto).strip()
    if not texto:
        raise ValueError("La reseña no puede estar vacía.")
    return predict_fn([texto])[0]

# Demo mínima
ejemplo = "El lugar es bonito, pero la atención fue muy lenta."
print("Predicción:", clasificar_resena(ejemplo))

### Costos a reportar

- Latencia media por reseña y tamaño de lote.
- Tamaño del modelo guardado.
- Memoria RAM/VRAM máxima.
- Hardware utilizado.
- Si se prueba cuantización o destilación, comparar costo y pérdida de métrica.

## Fase 6 — Monitoreo y mantenimiento

Monitorear distribución de clases predichas, confianza media, longitud de textos, vocabulario nuevo, origen/destino y desempeño cuando haya etiquetas. Simular drift con un subconjunto distinto por municipio, sitio, época o texto ruidoso y cuantificar la degradación.

In [ ]:
def distribucion_predicha(pred):
    return pd.Series(pred).value_counts(normalize=True).reindex(LABELS, fill_value=0)

baseline_distribution = distribucion_predicha(valid_pred_baseline)
submission_distribution = distribucion_predicha(submission_pred)
monitor = pd.DataFrame({
    "validación": baseline_distribution,
    "submission": submission_distribution,
})
monitor["diferencia_abs"] = (monitor["submission"] - monitor["validación"]).abs()
display(monitor)

# PENDIENTE: definir umbrales con evidencia del proyecto.
MAX_F1_DROP = 0.05
MAX_DISTRIBUTION_SHIFT = 0.15
print("Reentrenar si la caída de F1 supera", MAX_F1_DROP,
      "o si el cambio absoluto de alguna clase supera", MAX_DISTRIBUTION_SHIFT)

## Matriz transversal de QA y reproducibilidad

Completar una fila por fase; cada una debe conectar **riesgo → método QA → evidencia**.

| Fase | Riesgo | Método QA | Evidencia/archivo | Estado |
|---|---|---|---|---|
| 1. Negocio/datos | Desbalance, ruido, fuga | Auditorías y tablas cruzadas | Figuras/tablas EDA | En curso |
| 2. Datos | Solapamiento y truncamiento | Split agrupado, aserciones, análisis tokenizer | Salidas del notebook | En curso |
| 3. Modelo | Sobreajuste y baja trazabilidad | Early stopping y registro CSV | experimentos.csv | Pendiente |
| 4. Evaluación | Métrica insuficiente | F1 por clase, IC, errores, robustez | analisis_20_errores.csv | Pendiente |
| 5. Despliegue | Inferencia irreproducible | Carga desde modelo guardado y aserciones | submission_predictions.csv | Pendiente |
| 6. Monitoreo | Drift no detectado | Umbrales y simulación | Tabla de degradación | Pendiente |

## Bitácora obligatoria de IA generativa

Registrar cada uso relevante. El grupo debe comprender y validar todo texto o código aceptado.

In [ ]:
AI_LOG_PATH = ARTIFACTS_DIR / "bitacora_ia.csv"
AI_LOG_COLUMNS = [
    "fecha", "integrante", "herramienta", "proposito",
    "prompt_resumido", "aceptado", "corregido", "metodo_validacion"
]

if AI_LOG_PATH.exists():
    ai_log = pd.read_csv(AI_LOG_PATH)
else:
    ai_log = pd.DataFrame(columns=AI_LOG_COLUMNS)
    ai_log.to_csv(AI_LOG_PATH, index=False)

display(ai_log)
print("Bitácora:", AI_LOG_PATH.resolve())

## Cierre

### Conclusiones
**PENDIENTE:** responder la pregunta del proyecto y contrastar los resultados con los criterios de éxito.

### Limitaciones y trabajo futuro
**PENDIENTE:** datos, etiquetas, generalización, costo, sesgos y mejoras.

### Declaración de uso de IA generativa
**PENDIENTE:** resumir usos y enlazar la bitácora.

### Contribución individual

| Integrante | Datos | Modelado | Evaluación | Documentación | Evidencia |
|---|---:|---:|---:|---:|---|
| PENDIENTE | | | | | |

## Referencias iniciales

- Studer, S. et al. (2021). *Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology*. Machine Learning and Knowledge Extraction, 3(2), 392–413.
- Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
- Devlin, J. et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*. NAACL-HLT.
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed., borrador).

**PENDIENTE:** elegir APA o IEEE y mantener un único estilo.

## Lista final antes de entregar

- [ ] Resumen ≤ 200 palabras
- [ ] Objetivos de negocio, ML y económicos cuantificados
- [ ] EDA con tokens, desbalance, duplicados y casos ambiguos
- [ ] Split sin fuga y max_length justificado con porcentaje truncado
- [ ] Baseline y al menos 2 checkpoints comparados
- [ ] Al menos 5 corridas trazables
- [ ] Curvas y diagnóstico de sobreajuste
- [ ] Métrica oficial, F1 por clase/macro, matriz e incertidumbre
- [ ] Al menos 20 errores categorizados
- [ ] Robustez, explicabilidad, sesgos y privacidad
- [ ] Pipeline, demo, costo y envío reproducible
- [ ] Drift cuantificado y criterio de reentrenamiento
- [ ] QA con evidencia en todas las fases
- [ ] Bitácora de IA y contribuciones individuales
- [ ] Figuras/tablas numeradas y referencias correctas